In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


In [ ]:
df = pd.read_csv("../dataset/processed/cleaned_dataset.csv")


In [ ]:
df.head(5)


In [ ]:
df.head(20)


In [ ]:
print(df.nunique())

In [ ]:
df["Car_Age"] = 2025 - df["YOM"]
df["Mileage_per_Year"] = df["Millage(KM)"] / df["Car_Age"].replace(0, 1)

In [ ]:
df.info()


In [ ]:
df["Engine_Category"] = pd.cut(
    df["Engine (cc)"],
    bins=[0, 1000, 2000, 4000, float("inf")],
    labels=["Small", "Medium", "Large", "Very Large"]
)


In [ ]:
df.info()

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df["Year_Listed"] = df["Date"].dt.year
df["Month_Listed"] = df["Date"].dt.month

In [ ]:
binary_cols = ["POWER WINDOW", "POWER MIRROR", "POWER STEERING", "AIR CONDITION"]

mapping = {"Available": 1, "Not_Available": 0}

for col in binary_cols:
    df[col] = df[col].map(mapping)


In [ ]:
df.head(20)

In [ ]:
df["Leasing"] = df["Leasing"].apply(lambda x: 1 if "Leasing" in x else 0)

In [ ]:
df["Brand_Freq"] = df["Brand"].map(df["Brand"].value_counts())
df["Model_Freq"] = df["Model"].map(df["Model"].value_counts())


In [ ]:
df["Brand_AvgPrice"] = df.groupby("Brand")["Price"].transform("mean")
df["Model_AvgPrice"] = df.groupby("Model")["Price"].transform("mean")

In [ ]:
brand_freq_map = df["Brand"].value_counts().to_dict()
model_freq_map = df["Model"].value_counts().to_dict()

brand_avg_map = df.groupby("Brand")["Price"].mean().to_dict()
model_avg_map = df.groupby("Model")["Price"].mean().to_dict()

In [ ]:
luxury_brands = {"AUDI","BMW","MERCEDES","LEXUS","JAGUAR","LAND ROVER","PORSCHE","VOLVO"}
df["Luxury_Brand"] = df["Brand"].apply(lambda x: 1 if x.upper() in luxury_brands else 0)

In [ ]:
X = df.drop(columns=["Price", "Date", "Model", "Town"])

y = df["Price"]


In [ ]:
binary_cols = ["POWER WINDOW", "POWER MIRROR", "POWER STEERING", "AIR CONDITION", "Leasing", "Luxury_Brand"]
categorical = ["Brand", "Gear", "Fuel Type", "Condition", "Engine_Category"]
numeric = [col for col in X.columns if col not in categorical + binary_cols]


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("bin", "passthrough", binary_cols)
    ]
)

In [ ]:
X_transformed = preprocessor.fit_transform(X)


In [ ]:
num_features = numeric
bin_features = binary_cols
cat_features = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical)
feature_names = list(num_features) + list(bin_features) + list(cat_features)


In [ ]:
X_transformed_df = pd.DataFrame(
    X_transformed.toarray() if hasattr(X_transformed, "toarray") else X_transformed,
    columns=feature_names
)

In [ ]:
processed_df = X_transformed_df.copy()
processed_df["Price"] = y.values

In [ ]:
processed_df.to_csv(
    "../dataset/processed/preprocessed_car_price_dataset.csv",
    index=False
)
print("Preprocessed dataset saved successfully!")

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=42)
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = {}
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results[name] = {
        "MAE": mean_absolute_error(y_test, preds),
        "R2": r2_score(y_test, preds)
    }

print("\n--- Model Results ---")
print(results)

In [ ]:
best_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])


best_pipe.fit(X, y)

In [ ]:
def predict_from_row(index, pipeline):
    test_row = X.iloc[[index]]
    actual = y.iloc[index]
    predicted = pipeline.predict(test_row)[0]
    return actual, predicted

act, pred = predict_from_row(0, best_pipe)
print(f"Actual: {act}, Predicted: {pred:.2f}")

In [ ]:
def create_test_vehicle(yom, mileage, engine, brand, model, gear, fuel,condition,
                        aircon=1, ps=1, pm=1, pw=1, leasing=0, year_listed=2025, month_listed=1):
    car_age = 2025 - yom
    mileage_per_year = mileage / (car_age if car_age > 0 else 1)

    return pd.DataFrame([{
        "YOM": yom,
        "Millage(KM)": mileage,
        "Engine (cc)": engine,
        "Gear": gear,
        "Fuel Type": fuel,

        "Condition": condition,
        "AIR CONDITION": aircon,
        "POWER STEERING": ps,
        "POWER MIRROR": pm,
        "POWER WINDOW": pw,
        "Leasing": leasing,
        "Brand": brand,
        "Car_Age": car_age,
        "Mileage_per_Year": mileage_per_year,
        "Engine_Category": pd.cut([engine], bins=[0, 1000, 2000, 4000, float("inf")],
                                  labels=["Small", "Medium", "Large", "Very Large"])[0],
        "Year_Listed": year_listed,
        "Month_Listed": month_listed,
        "Brand_Freq": df["Brand"].value_counts().get(brand, 0),
        "Model_Freq": df["Model"].value_counts().get(model, 0),
        "Brand_AvgPrice": df.groupby("Brand")["Price"].mean().get(brand, df["Price"].mean()),
        "Model_AvgPrice": df.groupby("Model")["Price"].mean().get(model, df["Price"].mean()),
        "Luxury_Brand": 1 if brand.upper() in {"AUDI","BMW","MERCEDES","LEXUS",
                                               "JAGUAR","LAND ROVER","PORSCHE","VOLVO"} else 0
    }])


In [ ]:
# Example 1: Toyota AXIO 2017
sample1 = create_test_vehicle(
    yom=2012, mileage=120000, engine=1500,
    brand="TOYOTA", model="AQUA",
    gear="Auto", fuel="Petrol", condition="Used"
)

# Example 2: SUZUKI WAGONR
sample2 = create_test_vehicle(
    yom=2018, mileage=100000, engine=660,
    brand="SUZUKI", model="WAGON R",
    gear="Auto", fuel="Petrol",  condition="Used"
)

# Example 3: Suzuki Alto 2015
sample3 = create_test_vehicle(
    yom=2015, mileage=80000, engine=800,
    brand="SUZUKI", model="ALTO",
    gear="Manual", fuel="Petrol", condition="Used"
)

In [ ]:
pipe = Pipeline(steps=[("preprocessor", preprocessor),
                       ("model", RandomForestRegressor(n_estimators=200, random_state=42))])

pipe.fit(X, y)

for car in [sample1, sample2, sample3]:
    pred_price = pipe.predict(car)[0]
    print(f"Predicted Price: {pred_price*100000:,.0f} LKR")

In [ ]:
import joblib

# Save trained pipeline (preprocessor + model) with compression
joblib.dump(pipe, "../backend/artifacts/car_price_model.pkl", compress=3)
print("✅ Model pipeline saved successfully!")

# Save other artifacts normally
joblib.dump(brand_freq_map, "../backend/artifacts/brand_freq_map.pkl")
joblib.dump(model_freq_map, "../backend/artifacts/model_freq_map.pkl")
joblib.dump(brand_avg_map, "../backend/artifacts/brand_avg_map.pkl")
joblib.dump(model_avg_map, "../backend/artifacts/model_avg_map.pkl")

global_avg_price = df["Price"].mean()
joblib.dump(global_avg_price, "../backend/artifacts/global_avg_price.pkl")

print("✅ Feature maps saved successfully!")
